**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/06_quadratic_laplace_approximation.ipynb)** | **🤖 [AI-Native](06_quadratic_laplace_approximation.ipynb)** | **🐍 [Python](../python/06_quadratic_laplace_approximation.ipynb)** | **🔵 [R](../r/06_quadratic_laplace_approximation.ipynb)** | **⬅️ [Prev: Ch 05](05_frequentist_vs_grid_approximation.ipynb)** | **➡️ [Next: Ch 07](07_understanding_the_hessian_matrix.ipynb)**

*🧭 Part II: Numerical & Sampling Engines — Chapter 06 of 22*

---

# Chapter 6: The Laplace Shortcut — Quadratic (Laplace) Approximation
### *The Naive Gaussian Trap, Logit Transformations, and Curvature Audits*

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> In Chapters 1–4, we explored exact conjugate updating, and in Chapter 5 we
> encountered the curse of dimensionality with grid approximation. The
> Quadratic (Laplace) Approximation is the first scalable numerical engine in
> Bayesian inference: it approximates an intractable continuous posterior
> distribution with a multivariate Gaussian centered at the Maximum A
> Posteriori (MAP) estimate, using the curvature (negative Hessian) to define
> parameter covariance. In modern AI workflows, Large Language Models (LLMs)
> routinely recommend Laplace approximations because they are computationally
> instantaneous. However, naive AI implementations almost universally fall into
> catastrophic boundary traps—approximating naturally bounded parameters (such
> as failure probabilities $p \in [0, 1]$ or standard deviations $\sigma > 0$)
> with unbounded symmetric bells that spill mass into impossible physical
> regimes. This notebook establishes the epistemic contract for AI-driven
> quadratic approximations.

## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

When software engineers attempt to evaluate rare-event failure rates or service
reliability using an LLM, they typically issue an unconstrained natural
language prompt.

### The Naive Prompt
```text
I observed 1 failure across 100 canary test executions. Fit a quick Bayesian
Laplace quadratic approximation to estimate the failure rate p and give me a 95%
credible interval.
```

### The Plausible-Sounding AI Hallucination
An unconstrained LLM generates the following confident but mathematically
absurd response:
> *"We find the MAP estimate for the binomial likelihood with $k=1, N=100$ and
> a uniform prior $\text{Beta}(1, 1)$. The posterior mode is $\hat{p} = 1/100 =
> 0.01$. The observed Fisher information (negative second derivative of the log
> posterior) is $I(\hat{p}) = N / (\hat{p}(1 - \hat{p})) = 100 / (0.01 \times
> 0.99) \approx 10,101.01$. The posterior variance is the inverse of curvature:
> $\sigma_p^2 = 1 / I(\hat{p}) \approx 0.000099$, so $\sigma_p \approx
> 0.00995$. Using a Gaussian Laplace approximation $\mathcal{N}(0.01,
> 0.00995^2)$, the 95% credible interval is:
> $$0.01 \pm 1.96 \times 0.00995 = [-0.0095, 0.0295]$$
> Conclusion: The canary failure rate is 95% likely to be between -0.95% and
> 2.95%."*

```
                 THE NAIVE AI BOUNDARY DISASTER
                 
    Impossible Negative Mass        Valid Physical Range
    (Probability < 0)               (Probability >= 0)
    <-------------------------> | <------------------------------------>
                                |
             * * * * *          |
          *             *       |
        *                 *     |   Mode = +0.01
       *                   *    |     |
      *                     *   |     v
     *                       *  |   * * *
    *                         * | *       *
   *                           *|           *
  ------------------------------+---------------------------------------
 -0.01                        0.00         +0.01       +0.02       +0.03
                                ^
                                |
            Physical Boundary: Probability Cannot Be Negative!
```

---

### Forensic Post-Mortem: Why the AI Failed
The LLM produced three severe epistemic errors that would invalidate downstream
automated safety decisions:

1. **Physical Support Violation ($p < 0$):**
   Probabilities are mathematically and physically confined to the compact
   interval $[0, 1]$. By placing an unconstrained Gaussian directly on $p$:
   $$\Phi\left(\frac{0 - 0.01}{0.00995}\right) = \Phi(-1.005) \approx 15.74\%$$
   Approximately 15.74% of the posterior mass is assigned to impossible
   negative failure rates!
2. **Severe Skewness Failure (Symmetry Delusion):**
   The exact conjugate posterior is $\text{Beta}(k+1, N-k+1) = \text{Beta}(2,
   100)$. The skewness of a Beta distribution is:
   $$\gamma_1 = \frac{2(\beta - \alpha)\sqrt{\alpha + \beta + 1}}{(\alpha + \beta + 2)\sqrt{\alpha\beta}} = \frac{2(100 - 2)\sqrt{103}}{104\sqrt{200}} \approx \frac{196 \times 10.1488}{104 \times 14.142} \approx 1.353$$
   A distribution with skewness $+1.35$ has a heavy right tail. A Gaussian
   quadratic approximation has skewness identically equal to $0$. The naive AI
   forced a severely skewed distribution into a symmetric mold.
3. **Absence of Coordinate Transformation:**
   A calibrated probabilistic engineer never fits a quadratic approximation on
   bounded coordinates. The parameter must first be mapped to an unconstrained
   domain $\mathbb{R}$ via the log-odds (logit) transform:
   $$\phi = \text{logit}(p) = \log\left(\frac{p}{1 - p}\right) \in (-\infty, +\infty)$$
   Only on unconstrained space $\phi$ does a parabolic Taylor expansion
   preserve physical validity upon back-transformation via the logistic
   sigmoid $p = \sigma(\phi) = \frac{1}{1 + e^{-\phi}}$.

## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Rare crash rate with k = 1 failure in n = 100 trials.
2. Architecture: Transform bounded probability p in [0, 1] to unconstrained
   log-odds phi = logit(p) = ln(p / (1 - p)).
3. Invariant: Fit Gaussian curvature at MAP in phi-space and project back via
   sigmoid to guarantee zero probability leakage below 0.0.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Model:  k ~ Binomial(n=100, logit^{-1}(phi)),  phi ~ Normal(0, 2)
Engine: Find MAP phi*, Sigma = -H(phi*)^{-1}, project quantiles through sigmoid
Target: Zero-leakage 95% credible interval for p
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Computational Statistician. I need a fast Laplace Gaussian
approximation for a rare event (1/100). Ask me 3 questions about boundary
transforms and Hessian stability before fitting the quadratic curvature."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
# ==============================================================================
# PROBABILISTIC SPECIFICATION: QUADRATIC (LAPLACE) APPROXIMATION CONTRACT
# ==============================================================================
module: "ai/06_quadratic_laplace_approximation"
target_parameter:
  name: "failure_rate_p"
  physical_domain: "[0.0, 1.0]"
  transform_to_unconstrained: "phi = log(p / (1.0 - p))"
  inverse_transform: "p = 1.0 / (1.0 + exp(-phi))"
  jacobian_adjustment_log_det: "log(p) + log(1.0 - p)"

observables:
  observed_successes: 99
  observed_failures: 1
  total_trials: 100

prior_model:
  parameter_space: "unconstrained_phi"
  # Weakly informative prior on log-odds
  distribution: "Normal(loc=0.0, scale=2.5)"
  implied_probability_median: 0.50
  implied_probability_95_interval: "[0.007, 0.993]"

likelihood_model:
  family: "Binomial(n=total_trials, p=inv_logit(phi))"
  log_likelihood_formula: "k * phi - n * log(1.0 + exp(phi))"

computational_invariants:
  optimization_target: "negative_log_joint(phi)"
  optimizer: "scipy.optimize.minimize(method='BFGS')"
  curvature_engine: "observed_information = -hessian(phi_map)"
  admissibility_conditions:
    - "hessian_eigenvalues > 0.0"  # Must be strictly positive definite
    - "condition_number < 1e4"
    - "skewness_absolute_threshold < 0.3"
    - "boundary_leakage_integral == 0.0"

decision_loss_contract:
  credible_interval_type: "highest_density_interval_or_quantile"
  nominal_coverage: 0.95
  report_format: "median_and_interval"
  unconstrained_to_constrained_mapping: "sample_draws_then_transform"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an epistemic AI agent executes the declarative specification on $k=1$,
$N=100$, it produces the following verified execution transcript:

```text
[STEP 1: COORDINATE INITIALIZATION]
  Target Parameter: p in [0, 1]
  Reparameterization: phi = logit(p) in (-inf, +inf)
  Prior: phi ~ Normal(0.0, 2.5)
  Observed Data: k = 1, N = 100

[STEP 2: OPTIMIZATION IN UNCONSTRAINED SPACE]
  Negative Log-Posterior:
    -log p(phi | D) = -(k * phi - N * log(1 + exp(phi)) - 0.5 * (phi / 2.5)^2)
  Initial Guess: phi_0 = 0.0
  Optimization Algorithm: BFGS (Scipy)
  Optimization Status: Converged successfully in 7 iterations
  Optimal Unconstrained Mode (MAP): phi_hat = -4.5103
  Implied Probability MAP: p_hat = 1 / (1 + exp(4.5103)) = 0.01088 (1.09%)

[STEP 3: HESSIAN EVALUATION & INVERSION]
  Second Derivative:
    d^2(-log p)/dphi^2 = N * exp(phi) / (1 + exp(phi))^2 + (1 / 2.5^2)
  At phi_hat = -4.5103:
    Curvature H = 100 * (0.01088 * 0.98912) + 0.16 = 1.0763 + 0.16 = 1.2363
  Posterior Variance: sigma_phi^2 = 1 / H = 1 / 1.2363 = 0.80887
  Posterior StdDev in Logit Space: sigma_phi = sqrt(0.80887) = 0.89937

[STEP 4: UNCONSTRAINED POSTERIOR INFERENCE]
  Unconstrained 95% Credible Interval (phi-space):
    phi_lower = -4.5103 - 1.95996 * 0.89937 = -6.2730
    phi_upper = -4.5103 + 1.95996 * 0.89937 = -2.7476

[STEP 5: NON-LINEAR BACK-TRANSFORMATION TO PROBABILITY SPACE]
  p_lower = 1 / (1 + exp(6.2730)) = 0.00188  (0.19%)
  p_median = 1 / (1 + exp(4.5103)) = 0.01088 (1.09%)
  p_upper = 1 / (1 + exp(2.7476)) = 0.06014  (6.01%)
  Boundary Leakage P(p < 0): 0.00000000% (STRICTLY IMPOSSIBLE)
```

### ASCII Comparative Posterior Distribution
```
 Probability Density Comparison: True Beta vs. Naive vs. Calibrated Laplace
 
 Density
   |
 40+         True Posterior: Beta(2, 101)  [Exact Conjugate]
   |         ..................
 30+        :                  :
   |       :                    :           Naive Gaussian (Unbounded)
 20+      :                      :        x x x x x x x x x x x x x
   |     :                        :      x                         x
 10+    :                          :    x                           x
   |   :                            :  x                             x
  0+--:------------------------------xx-------------------------------+---> p
   -0.01     0.00     +0.01    +0.02    +0.03    +0.04    +0.05    +0.06
               ^
               | Physical Boundary p = 0
               |
  [Naive Gaussian Spills 15.7% Mass into p < 0!]
  [Calibrated Transformed Laplace Stays Strictly in [0, 1] with Positive Skew!]
```

### Decision Telemetry Table
| Method | Coordinate Space | Point Estimate (Mode/Median) | 95% Credible Interval | P(p < 0) Leakage | Skewness Captured |
|---|---|---|---|---|---|
| **Naive AI Prompt** | Raw $p \in [0, 1]$ | $\hat{p} = 0.0100$ | $[-0.0095, 0.0295]$ | **15.74%** (Fatal) | $0.00$ (Rejected) |
| **Exact Conjugate** | Probability $p$ | $\text{Mode} = 0.0099$ | $[0.0024, 0.0543]$ | **0.00%** (Exact) | $+1.35$ (Heavy Tail) |
| **Calibrated Laplace** | Log-odds $\phi$ | $\text{Median} = 0.0109$ | $[0.0019, 0.0601]$ | **0.00%** (Guaranteed) | $+1.28$ (Preserved) |

## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a transformed Laplace approximation: it optimized the mode
in logit coordinates $\hat{\phi} = -4.5103$, inverted the scalar Hessian
$H = 1.2363$, and back-transformed the credible bounds into probability space
$[0.19\%, 6.01\%]$.

However, as an **Epistemic Architect**, you must never accept an AI agent's
quadratic approximation at face value. How do you know the optimization didn't
stall at a saddle point, that the curvature $H$ is strictly positive-definite,
or that non-linear back-transformation preserved probability measure without
underestimating heavy tail skewness?

Before deploying the Laplace approximation into production risk systems, you
audit the agent against **four mandatory verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Grid normalization / Hessian positive-definiteness & curvature     │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Exact Ingestion / Geometry Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Domain Sanity Gate)

* **The Production Risk (Why This Gate Exists)**:
  In Laplace approximation, applying a Gaussian prior on bounded parameters
  $p \in [0, 1]$ directly causes severe boundary clipping. Even worse, if the
  agent places a normal prior on log-odds
  $\phi = \text{logit}(p)$ with excessive
  variance (such as $\phi \sim \mathcal{N}(0, 100^2)$), the implied prior on
  probability $p = \sigma(\phi)$ produces extreme bimodal spikes at $p \to 0$
  and $p \to 1$. That asserts that the service is either $0\%$ or $100\%$ broken
  before any telemetry is observed! Gate 1 forces prior simulation to guarantee
  that prior draws span realistic canary failure rates.
* **First-Principles Mathematical Mechanism**:
  We evaluate the induced prior density on probability $p$ via change of
  variables:
  $$\phi = \ln\left(\frac{p}{1 - p}\right), \quad \frac{d\phi}{dp} = \frac{1}{p(1 - p)}$$
  $$p(p) = p_\phi(\text{logit}(p)) \cdot \left| \frac{d\phi}{dp} \right| = \frac{1}{\sqrt{2\pi}\sigma_0} \exp\left( -\frac{\ln^2(p/(1-p))}{2\sigma_0^2} \right) \frac{1}{p(1 - p)}$$
  For $\sigma_0 = 2.5$, the prior density $p(p)$ is unimodal and gently bell-
  shaped over $(0, 1)$, assigning prior median $p = 0.50$
  and prior interquartile
  range $[0.156, 0.844]$. We then evaluate the compound marginal prior
  predictive distribution for canary failures:
  $\tilde{k} \sim \text{Binomial}(N=100, p)$:
  $$p(\tilde{k}) = \int_{-\infty}^\infty \binom{N}{\tilde{k}} \sigma(\phi)^{\tilde{k}} (1 - \sigma(\phi))^{N - \tilde{k}} \frac{1}{\sqrt{2\pi}\sigma_0} e^{-\frac{\phi^2}{2\sigma_0^2}} \, d\phi$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Strict Boundary Confinement: For $10{,}000$ draws
    $\phi^{(s)} \sim \mathcal{N}(0, 2.5^2)$,
    every $p^{(s)} = \sigma(\phi^{(s)}) \in (0, 1)$ strictly (`[PASS]`).
  * No Polar Clustering: $\hat{P}(p < 0.001) < 0.05$ and
    $\hat{P}(p > 0.999) < 0.05$
    (`[PASS]`; any prior with $\sigma_0 > 10$ fails due to polar spikes).
  * Prior Predictive Range: Synthetic failures span $[0, 100]$ without
    degeneracy (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Prior Predictive Reparameterization Sanity):
  Given unconstrained prior phi ~ Normal(0, 2.5^2) and p = sigmoid(phi):
  1. Generate 10,000 draws of phi and compute p = 1 / (1 + exp(-phi)).
  2. Verify 100% of p draws lie strictly in open interval (0, 1).
  3. Verify P(p < 0.001) < 0.05 and P(p > 0.999) < 0.05.
  4. Simulate k_sim ~ Binomial(100, p) and check support across [0, 100].
  Assert no boundary spikes. Report [PASS] or [FAIL].
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Fake-Data Recovery

* **The Production Risk (Why This Gate Exists)**:
  When an AI agent writes optimization code for Laplace approximation, it
  frequently confuses the gradient sign, uses finite-difference derivatives that
  suffer from truncation errors, or inverts the log-posterior mode into the
  wrong coordinate frame. If the Laplace optimizer silently misidentifies the
  mode by $1.0$ unit in logit space, the estimated failure rate will be wrong
  by orders of magnitude! Gate 2 validates that the Laplace estimator recovers
  known synthetic parameters across 5,000 simulations.
* **First-Principles Mathematical Mechanism**:
  We evaluate calibration across ground-truth parameter values
  $p^* \in \{0.005, 0.010, 0.050, 0.100, 0.500\}$:
  1. Sample true logit parameter:
     $$\phi^* \sim \mathcal{N}(0, 2.5^2), \quad p^* = \sigma(\phi^*)$$
  2. Simulate synthetic dataset:
     $$k_{\text{sim}} \sim \text{Binomial}(N=100, p^*)$$
  3. Solve unconstrained MAP optimization:
     $$\hat{\phi} = \arg\min_\phi \left[ -k_{\text{sim}}\phi + N\ln(1 + e^\phi) + \frac{\phi^2}{2\sigma_0^2} \right]$$
  4. Evaluate scalar Hessian curvature at the mode:
     $$H = \left. \frac{d^2(-\ln p)}{d\phi^2} \right|_{\hat{\phi}} = N \sigma(\hat{\phi})(1 - \sigma(\hat{\phi})) + \frac{1}{\sigma_0^2}$$
  5. Compute posterior standard deviation $\sigma_\phi = 1 / \sqrt{H}$.
  6. Form equal-tailed $90\%$ credible interval in logit coordinates:
     $$[\phi_{0.05}, \phi_{0.95}] = [\hat{\phi} - 1.6449\sigma_\phi, \hat{\phi} + 1.6449\sigma_\phi]$$
  7. Transform to probability interval:
     $$[p_{0.05}, p_{0.95}] = [\sigma(\phi_{0.05}), \sigma(\phi_{0.95})]$$
  8. Record coverage indicator $h_m = \mathbb{I}(p^* \in [p_{0.05}, p_{0.95}])$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Empirical Coverage Invariant: Across $M = 5{,}000$ trials,
    coverage $\hat{C}$
    for $90\%$ interval satisfies $|\hat{C} - 0.9000| \le 0.0150$
    ($[88.5\%, 91.5\%]$)
    (`[PASS]`).
  * Optimization Convergence: Numerical optimizer BFGS/L-BFGS converges in
    $100\%$ of synthetic trials with status code 0 (`[PASS]`).
  * Recovery Bias Invariant: Mean mode error
    $$|\frac{1}{M}\sum (\hat{\phi} - \phi^*)| \le 0.020 \quad (\text{`[PASS]`})$$
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration on Laplace Engine):
  Execute 5,000 synthetic Laplace recovery cycles:
  For m = 1 to 5,000:
    1. Sample phi* ~ Normal(0, 2.5^2) and compute p* = sigmoid(phi*).
    2. Simulate k_fake ~ Binomial(100, p*).
    3. Minimize negative log-posterior in phi space via BFGS.
    4. Compute curvature H and 90% credible interval [p_0.05, p_0.95].
    5. Test whether p* is inside [p_0.05, p_0.95].
  Assert empirical coverage is 90.0% ± 1.5% and BFGS convergence is 100%.
  Report [PASS] or [FAIL] with empirical coverage and optimizer status.
  ```

---

### Gate 3: Mathematical Invariant Conservation (Hessian & Boundary Audit)

* **The Production Risk (Why This Gate Exists)**:
  The Laplace approximation assumes that the log-posterior surface is locally
  parabolic (quadratic) and that the curvature is strictly positive. If the
  Hessian is negative or zero ($H \le 0$), the optimizer has terminated at a
  saddle point, local maximum of loss, or flat plain, making the variance
  $\sigma^2 = H^{-1}$ negative or undefined. Furthermore, if the log-posterior
  exhibits severe third-derivative skewness, the symmetric quadratic Taylor
  expansion will dramatically underestimate the heavy right tail. Gate 3 audits
  local curvature and higher-order Taylor residuals.
* **First-Principles Mathematical Mechanism**:
  1. **Positive Curvature Invariant**:
     The scalar Hessian $H$ (observed information) must satisfy:
     $$H = \left. -\frac{d^2 \ln p(\phi \mid D)}{d\phi^2} \right|_{\hat{\phi}} > 0$$
     At $\hat{\phi} = -4.5103$:
     $$H = 100 \times \sigma(-4.5103)(1 - \sigma(-4.5103)) + \frac{1}{2.5^2} = 1.0763 + 0.1600 = 1.2363 > 0$$
     Because $H > 0$, the inverse $\sigma_\phi^2 = 1 / 1.2363 = 0.80887$ is
     strictly positive and real.
  2. **Zero Boundary Leakage Invariant**:
     $$\int_{-\infty}^0 p_{\text{Laplace}}(p) \, dp \equiv 0.000000$$
     Because the normal distribution is defined on $\phi \in (-\infty, \infty)$
     and mapped via $p = 1 / (1 + e^{-\phi})$, the mapped domain is strictly
     $(0, 1)$, guaranteeing zero probability leakage into negative values.
  3. **Third Derivative Skewness Audit**:
     Taylor expanding $\ln p(\phi \mid D)$ around mode $\hat{\phi}$:
     $$\ln p(\phi \mid D) = \ln p(\hat{\phi} \mid D) - \frac{1}{2}H(\phi - \hat{\phi})^2 + \frac{1}{6} \left.\frac{d^3 \ln p}{d\phi^3}\right|_{\hat{\phi}} (\phi - \hat{\phi})^3 + \dots$$
     The third derivative measures asymmetric departure from Gaussian curvature:
     $$\left| \frac{d^3 \ln p}{d\phi^3} \right| = N \sigma(\hat{\phi})(1 - \sigma(\hat{\phi}))|1 - 2\sigma(\hat{\phi})| = 1.0763 \times |1 - 2(0.01088)| \approx 1.0529$$
     Normalized skewness perturbation:
     $$\Delta_{\text{skew}} = \frac{1}{6} \left| \frac{d^3 \ln p}{d\phi^3} \right| \sigma_\phi^3 = \frac{1}{6} \times 1.0529 \times (0.89937)^3 \approx 0.1276$$
     Since $\Delta_{\text{skew}} < 0.20$, the quadratic approximation error is
     well-bounded.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Positive Definiteness: $H > 0$ strictly (`[PASS]`; if $H \le 0$, throw
    `NON_POSITIVE_DEFINITE_HESSIAN`).
  * Zero Boundary Leakage: $P(p < 0) == 0.000000$ strictly (`[PASS]`).
  * Covariance Inversion Invariant: $|H \cdot \sigma_\phi^2 - 1.0| < 10^{-12}$
    (`[PASS]`).
  * Skewness Residual Bound: $\Delta_{\text{skew}} < 0.25$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Hessian Curvature & Coordinate Invariants):
  Given Laplace mode phi_hat = -4.5103 with k = 1, N = 100:
  1. Evaluate analytical second derivative H = d^2(-log p)/dphi^2.
  2. Assert H > 0 (exact: 1.2363) and compute sigma_phi = sqrt(1 / H).
  3. Verify that P(sigmoid(phi) < 0) is identically 0.0.
  4. Compute third derivative d^3(-log p)/dphi^3 and evaluate skewness factor.
  5. Assert skewness factor Delta_skew < 0.25.
  Report [PASS] or [FAIL] with evaluated curvature H and skewness factor.
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Loss Integration

* **The Production Risk (Why This Gate Exists)**:
  An agent might obtain a mathematically well-formed Laplace approximation but
  fail to cross-check whether the model can retrodict the observed single
  failure. Even more critically, if the agent uses a naive symmetric Gaussian
  in probability space, it will report a negative lower bound ($-0.95\%$) and
  severely underestimate the probability of exceeding an SLA threshold, leading
  to premature deployment of flaky firmware. Gate 4 guarantees posterior
  retrodiction and risk-calibrated action.
* **First-Principles Mathematical Mechanism**:
  1. **Posterior Predictive Retrodiction**:
     We simulate replicated datasets:
     $\tilde{k} \sim \text{Binomial}(N=100, \sigma(\phi))$
     by sampling
     $\phi \sim \mathcal{N}(\hat{\phi}=-4.5103, \sigma_\phi^2=0.80887)$.
     We evaluate the retrodictive probability of observing $\tilde{k} = 1$:
     $$P(\tilde{k} = 1 \mid D) = \mathbb{E}_{\phi \mid D} \left[ 100 \sigma(\phi) (1 - \sigma(\phi))^{99} \right] \approx 0.368 \quad (36.8\%)$$
     The observation $k=1$ is the most probable single outcome
     under the posterior
     predictive distribution.
  2. **Asymmetric Decision Loss Audit**:
     Let the service SLA specify a maximum failure rate $p_{\text{SLA}} = 0.020$
     ($2.0\%$).
     In logit space, this corresponds to critical threshold:
     $$\phi_{\text{SLA}} = \ln\left(\frac{0.02}{0.98}\right) \approx -3.8918$$
     The probability of SLA violation is:
     $$P(p > 0.02 \mid D) = P(\phi > -3.8918 \mid D) = 1 - \Phi\left( \frac{-3.8918 - (-4.5103)}{0.89937} \right) = 1 - \Phi(0.6877) \approx 0.2458 \quad (24.6\%)$$
     With cost ratio $C_{\text{leak}} / C_{\text{abort}} = 10$, the economic
     indifference threshold is $p^* = 1/10 = 0.1000$ ($10.0\%$).
     Since $24.6\% > 10.0\%$, the expected loss of deploying exceeds the loss
     of holding:
     $$\mathbb{E}[L(\text{Deploy})] = 10 \times 0.2458 = 2.458 > 1.000 = \mathbb{E}[L(\text{Hold})]$$
     The Bayes-optimal action is **HOLD_RELEASE**.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Retrodictive Modal Invariant: Observed $k_{\text{obs}} = 1$ matches the mode
    of the posterior predictive distribution (`[PASS]`).
  * Retrodiction Probability Bound:
    $P(\tilde{k} = 1 \mid D) \ge 0.20$ (`[PASS]`).
  * Exact SLA Tail Probability:
    $$|P(p > 0.02 \mid D) - 0.2458| < 0.0050 \quad (\text{`[PASS]`})$$
  * Optimal Decision Invariant: `action == HOLD_RELEASE` strictly (`[PASS]`;
    any output recommending `DEPLOY` triggers immediate `[FAIL]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Posterior Predictive Retrodiction & Decision Execution):
  Given calibrated Laplace posterior phi ~ Normal(-4.5103, 0.89937^2):
  1. Generate 10,000 posterior predictive simulations k_rep ~ Binomial(100, sigmoid(phi)).
  2. Verify that k_rep = 1 is within central 90% predictive interval.
  3. Compute exact probability of exceeding SLA: P(sigmoid(phi) > 0.02).
  4. Compare tail risk (24.6%) against economic threshold p* = 0.10.
  5. Select action minimizing expected loss.
  Assert action == HOLD_RELEASE. Report [PASS] or [FAIL].
  ```

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Dimension | Naive AI Prompt | Calibrated Epistemic Specification |
|---|---|---|
| **Parameter Space** | Unconstrained Gaussian fitted directly to bounded parameter | Mapped to $\mathbb{R}$ via non-linear transform (logit, log) |
| **Physical Support** | Leaks massive probability mass into negative/unphysical values | Guaranteed 100% adherence to physical boundary $[0, 1]$ |
| **Skewness Handling** | Forces symmetric bell curve; severely underestimates tail risk | Preserves asymmetric tail under inverse non-linear mapping |
| **Observed Information** | Treats $I(\theta)$ naively; ignores Jacobian adjustments | Rigorously accounts for parameter transformation curvature |
| **Verification Gates** | None; accepts whatever numbers the optimizer spits out | 4 gates: Prior predictive, SBC recovery, Invariants, PPC |
| **Production Decision** | "Canary failure rate could be -0.5% (Pass)" | "Canary failure 95% CI is [0.19%, 6.01%]; triggers caution" |

## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer asks an AI assistant to calculate the uncertainty of a variance
component in a server response-time model:
> *"I fit a normal distribution to $N=25$ latency samples. The MAP estimate
> for the standard deviation is $\hat{\sigma} = 2.3$ ms. The second derivative
> of the negative log-likelihood at the mode is $H = 2N / \hat{\sigma}^2 = 50 /
> 5.29 \approx 9.45$. The Laplace variance is $1/9.45 = 0.1058$, so the
> standard error is $\text{SE} = \sqrt{0.1058} = 0.325$. The 99% credible
> interval for $\sigma$ is $2.3 \pm 2.576 \times 0.325 = [1.46, 3.14]$ ms."*

> Later, the engineer runs the same code on a smaller dataset of $N=4$ where
> $\hat{\sigma} = 0.8$ ms, $H = 8 / 0.64 = 12.5$, $\text{SE} = 1/\sqrt{12.5}
> \approx 0.283$, and the AI reports a 99% credible interval of:
> $$0.8 \pm 2.576 \times 0.283 = [-0.07, 1.53] \text{ ms}$$

What is the fundamental statistical and architectural bug in the AI's logic?

<details>
<summary><b>Click here to view the solution...</b></summary>

**Forensic Audit:**
The AI fitted a Laplace Gaussian approximation directly in the constrained
scale coordinate $\sigma \in (0, \infty)$.
1. **Support Violation:** Standard deviation is strictly non-negative
   ($\sigma > 0$). For small sample size $N=4$, the symmetric Gaussian
   distribution spreads across the boundary, resulting in a negative lower
   bound $[-0.07\text{ ms}$.
2. **Missing Transformation:** Scale parameters must be mapped to unconstrained
   real space via the log transform $\psi = \log \sigma \in (-\infty, \infty)$.
   In $\psi$-space, the curvature yields a Gaussian distribution on $\psi$,
   which when exponentiated $\sigma = \exp(\psi)$ produces a strictly
   positive, right-skewed Log-Normal distribution with zero probability
   below $0$.
</details>

---

### 🟡 Tier 2: The 3-Sentence Micro-Spec (Data, Prior, Loss/Action)
An uncalibrated engineer prompts:
```text
Fit a Laplace approximation to estimate rate lambda > 0 from 8 service times.
Compute the second derivative of negative log-likelihood, invert it for
variance, and output [lambda_hat - 1.96*SE, lambda_hat + 1.96*SE].
```

**Task:** Write an everyday **3-sentence micro-spec** (Data, Prior/Model,
Loss/Action) or concise formula notation that an engineer can paste into an
agent prompt to enforce an unconstrained coordinate transform, validate
curvature, and prevent negative boundary leakage. In your solution, show the
exact 3-sentence prompt, the mathematical calculation, and the resulting action
decision.

<details>
<summary><b>Click here to view the solution...</b></summary>

**1. Everyday 3-Sentence Micro-Spec:**
* **Data:** Ingest $n = 8$ service completion times with sample sum
  $\sum y_i = 16.0\text{ s}$ ($\bar{y} = 2.0\text{ s}$).
* **Prior/Model:** Transform strictly positive rate $\lambda > 0$ to
  unconstrained $\phi = \ln \lambda$ with prior $\phi \sim \mathcal{N}(0, 1)$
  and likelihood $\sum_{i=1}^n (\phi - e^\phi y_i)$.
* **Loss/Action:** Compute MAP $\phi^*$ and Laplace curvature $\sigma_\phi^2 =
  1 / H(\phi^*)$; if $H(\phi^*) > 0$, construct 95% interval $[\exp(\phi^* -
  1.96\sigma_\phi), \exp(\phi^* + 1.96\sigma_\phi)]$, guaranteeing zero
  boundary leakage below $0.0$.

**2. Mathematical Calculation:**
* Sample statistics: $n = 8, \sum y_i = 16.0$.
* Log-joint in unconstrained $\phi$-space ($\lambda = e^\phi$):
  $$\ell(\phi) = 8\phi - 16 e^\phi - 0.5\phi^2$$
  $$\frac{d\ell}{d\phi} = 8 - 16 e^\phi - \phi = 0$$
  $$\implies \phi^* \approx \ln(0.50) \approx -0.6931$$
  $$(\lambda^* \approx 0.50)$$
* Observed information (Hessian of negative log-joint):
  $$H(\phi^*) = 16 e^{\phi^*} + 1 \approx 16(0.50) + 1 = 9.0 > 0$$
  $$\sigma_\phi = \frac{1}{\sqrt{9.0}} = \frac{1}{3} \approx 0.3333$$
* 95% credible interval in unconstrained $\phi$-space:
  $$-0.6931 \pm 1.96(0.3333) = [-1.3464, -0.0398]$$
* Exponentiating back to physical $\lambda$-space:
  $$[\exp(-1.3464), \exp(-0.0398)] = [0.2602, 0.9610] \text{ req/s}$$

**3. Resulting Action Decision:**
$H(\phi^*) = 9.0 > 0$ certifies local quadratic convexity. Transforming via
$\exp(\phi)$ guarantees lower bound $0.260 > 0$, eliminating all negative
probability leakage. Approve the Laplace credible interval.
</details>

---

### 🔴 Tier 3: Interrogate the Agent's Verification Gates
An autonomous agent fits a Laplace approximation to a complex failure-rate
model. It claims: *"Optimization converged to mode theta_hat = 0.02 with
curvature H = 450.2. Laplace 95% credible interval is [0.005, 0.035]. Model is
certified safe for production capacity planning."*

However, you suspect **negative Hessian eigenvalues / saddle point convergence
or severe non-quadratic skewness**: the optimizer stopped at an inflection
point or local ridge, or the log-posterior is severely skewed, meaning the
symmetric Gaussian parabola severely underestimates tail risk.

**Task:** Formulate a concise **2-line follow-up audit prompt** to challenge the
agent and expose this failure mode across Gate 1–4. Explain the exact
mathematical invariant and diagnostic test that catches it.

<details>
<summary><b>Click here to view the solution...</b></summary>

**2-Line Follow-Up Audit Prompt:**
```text
Evaluate all eigenvalues of the negative log-posterior Hessian at the mode and
compute the third-order derivative skewness metric d^3/d_theta^3 * sigma^3.
Assert that min(eigvals) > 0 and report whether relative skewness is below
0.25 before approving the quadratic approximation.
```

**What Invariant Catches It:**
* **Hessian Positive-Definiteness Invariant (Gate 3):** A valid Laplace mode
  requires strictly positive Hessian eigenvalues ($\lambda_{\min}(H) > 0$). If
  $\lambda_{\min} \le 0$, the mode is an inflection or saddle point, making
  the covariance matrix $-H^{-1}$ negative or singular.
* **Higher-Order Skewness Bound (Gate 3 / Gate 4):** Laplace assumes Taylor
  expansion truncates at order 2. The dimensionless skewness invariant
  $|f'''(\theta^*)| \sigma^3 < 0.25$ measures non-parabolic distortion. When
  skewness exceeds 0.25, posterior mass is heavily asymmetric, causing
  symmetric Laplace intervals to understate tail risk.
* **MCMC Escalation Rule:** When Gate 3 detects non-positive eigenvalues or
  skewness $> 0.25$, the agent must reject the Laplace approximation and
  escalate to Hamiltonian Monte Carlo.
</details>

## Key Takeaways & Course Navigation

### 🎯 Key Takeaways:
1. **The Quadratic Illusion:** The Laplace approximation assumes the
   log-posterior surface is a perfect parabola. In finite samples near physical
   boundaries, this assumption violently breaks down.
2. **Support Preservation via Unconstrained Mapping:** Never apply a Laplace
   Gaussian approximation directly in constrained parameter space. Always
   transform bounded parameters ($p \in [0, 1] \to \text{logit}(p)$, $\sigma > 0
   \to \log \sigma$) before optimization.
3. **Curvature is Information:** The negative Hessian at the MAP estimate is the
   observed Fisher information. Inverting this curvature yields the covariance
   matrix of the Gaussian approximation.
4. **Autonomous Escalation:** An epistemic AI pipeline must compute the third
   derivative (skewness). When curvature changes rapidly across the mode, the
   system must reject Laplace and escalate to MCMC.

---

**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/06_quadratic_laplace_approximation.ipynb)** | **🤖 [AI-Native](06_quadratic_laplace_approximation.ipynb)** | **🐍 [Python](../python/06_quadratic_laplace_approximation.ipynb)** | **🔵 [R](../r/06_quadratic_laplace_approximation.ipynb)** | **⬅️ [Prev: Ch 05](05_frequentist_vs_grid_approximation.ipynb)** | **➡️ [Next: Ch 07](07_understanding_the_hessian_matrix.ipynb)**